# pRF
samples energy outputs from the steerable pyramid model using population receptive field (pRF) data for specific brain regions (e.g., V1, V2) and saves the results

**For Iking**

I run the code on my visual studio code lenovo laptop- have all files in a usb drive.
however all the necessery files are in our lab's google drive folder-

Under My Drive-> V1_Drift:
1. prfsample_expand- my outputs from running the prf sampling for subject 1 
2. NSD-> pyramind_expand - steerable pyramind outputs for images 0-15 (in the driver its 72k images so 4 TeraByte)

Rest of the stuff i think you know- each subjects files of betas and prf and stuff are in its own folder under NSD folder, and the expirement itself is in NSD->nsddata->expirements->nsd.

i run with 10 cores and 10 images in a batch and for some of the participant its for 10k images (those with 40 session).



In [ ]:
import os
import numpy as np
import nibabel as nib  # Used to load .nii.gz neuroimaging files
import scipy.io as sio  # For loading MATLAB .mat files
import h5py  # For working with HDF5 files
import gc  # For manual garbage collection to free memory
from joblib import Parallel, delayed  # For parallel processing
from tqdm import tqdm  # For progress bars
import tempfile, shutil  # For safely saving output files

# -----------------------------------------------------------------------
# Paths config
# -----------------------------------------------------------------------
nsd_folder = r"D:\NSD"  # Path to NSD dataset
pyramid_folder = r"D:\NSD\pyramid_expand"  # Folder with steerable pyramid outputs
prf_folder = r"C:\nsd\prfsample_expand"  # Folder to save output of pRF sampling

# -----------------------------------------------------------------------
# General settings
# -----------------------------------------------------------------------
interp_img_size = 714  # Interpolated image size used in model
background_size = 1024  # Size of the padded image canvas
img_scaling = 0.5  # Image scaling factor
num_levels = 7  # Number of pyramid spatial frequency levels
num_orientations = 8  # Number of orientation filters
pix_per_deg = (
    img_scaling * 714 / 8.4
)  # Conversion factor: pixels per degree of visual angle
deg_per_pix = 8.4 / (714 * img_scaling)  # Degrees per pixel

roi_names = ["V1v", "V1d", "V2v", "V2d", "V3v", "V3d", "hV4"]
test_mode = False  # Toggle test mode (fewer images) for debugging


# ------------------------------------------------------------------------------
# ROI Indexing Clarification
# ------------------------------------------------------------------------------
# In the pRF visual ROI label file, the mapping is as follows:
#   0 → Unknown
#   1 → V1v
#   2 → V1d
#   3 → V2v
#   4 → V2d
#   5 → V3v
#   6 → V3d
#   7 → hV4
#
# In this code, we define:
#   roi_map = {
#       1: [0, 1],  # V1v + V1d
#       2: [2, 3],  # V2v + V2d
#       ...
#   }
#
# IMPORTANT: The entries [0, 1] in roi_map refer to INDEXES (0-based),
# and we apply `+1` to match the ROI labels from the file.
# Therefore:
#   roinum = 0 → visRoiData == 1 → V1v
#   roinum = 1 → visRoiData == 2 → V1d
#
# This means when saved results show "roi_0" or "roi_1", they correspond to
# V1v and V1d respectively — **NOT** Unknown and V1v.
# ------------------------------------------------------------------------------


# -----------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------


def load_nifti(filepath):
    """Load a NIfTI file if it exists; return None otherwise."""
    return nib.load(filepath).get_fdata() if os.path.exists(filepath) else None


def load_processed_images_log(log_path):
    """Read log file of already processed image indices. Returns a set."""
    if os.path.exists(log_path):
        with open(log_path, "r") as log_file:
            return set(map(int, filter(str.isdigit, log_file.read().splitlines())))
    return set()


def process_single_image(imgNum, roiGaus, num_levels, num_orientations):
    """
    Process a single image by applying ROI-specific 2D Gaussian pRF filters
    to outputs from a steerable pyramid model of the image.

    Args:
        imgNum (int): The index of the image being processed (1-based).
        roiGaus (dict[int, np.ndarray]): Dictionary mapping ROI index to
            an array of shape (n_voxels, H, W) with precomputed 2D Gaussian
            filters for each voxel in that ROI.
        num_levels (int): Number of spatial frequency levels in the pyramid.
        num_orientations (int): Number of orientation channels per level.

    Returns:
        tuple:
            - imgNum (int): The image number that was processed.
            - prfSampleLev_partial (dict[int, np.ndarray]): For each ROI,
              an array of shape (n_voxels, num_levels+2) representing the
              sumOri activation (orientation-agnostic).
            - prfSampleLevOri_partial (dict[int, np.ndarray]): For each ROI,
              an array of shape (n_voxels, num_levels+2, num_orientations)
              representing orientation-specific responses.

    Example:
        >> imgNum = 124
        >> roiGaus = {0: np.random.rand(150, 512, 512)}  # One ROI with 150 voxels
        >> process_single_image(imgNum, roiGaus, 7, 8)
    """
    pyramid_filename = os.path.join(pyramid_folder, f"pyrImg{imgNum - 1}.mat")
    try:
        data = sio.loadmat(pyramid_filename)
        sumOri = data["sumOri"]  # List of 2D arrays; [num_levels+2][H, W]
        modelOri = data["modelOri"][
            0
        ]  # List of lists; [num_levels][num_orientations][H, W]
    except Exception as e:
        print(f"Failed to load image {imgNum}: {e}")
        return imgNum, {}, {}

    # Initialize containers for each ROI's responses
    prfSampleLev_partial = {}  # Stores orientation-summed projection results
    prfSampleLevOri_partial = {}  # Stores orientation-specific projection results

    for roinum, G_all in roiGaus.items():
        n_voxels = G_all.shape[0]  # Number of voxels in this ROI
        # Initialize output arrays
        prf_levels = np.zeros((n_voxels, num_levels + 2), dtype=np.float32)
        prf_oris = np.zeros(
            (n_voxels, num_levels + 2, num_orientations), dtype=np.float32
        )

        # Project Gaussian kernels over sumOri (orientation-summed response)
        for ilev in range(min(num_levels + 2, len(sumOri))):
            # sumOri[ilev]: shape (H, W)
            # G_all: shape (n_voxels, H, W)
            # Output: dot product of each voxel's Gaussian with the image
            prf_levels[:, ilev] = np.tensordot(
                G_all, sumOri[ilev], axes=([1, 2], [0, 1])
            )

        # Project Gaussian kernels over orientation-specific modelOri
        for ilev in range(min(num_levels, len(modelOri))):
            for iori in range(num_orientations):
                # Multiply each voxel's filter with the oriented map and sum over H, W
                prf_oris[:, ilev, iori] = np.sum(
                    modelOri[ilev][iori] * G_all, axis=(1, 2)
                )

        # Store results for this ROI
        prfSampleLev_partial[roinum] = prf_levels
        prfSampleLevOri_partial[roinum] = prf_oris

    return imgNum, prfSampleLev_partial, prfSampleLevOri_partial


def prf_sample_model_expand(isub, visualRegion, batch_size=100):
    """
    Main function to compute pRF samples by applying ROI Gaussians to steerable pyramid outputs.

    Args:
        isub: Subject index (1-based).
        visualRegion: Visual region group ID (1 to 4).
        batch_size: Number of images to process in parallel per batch.
    """
    print(f"\n-- Running pRF Sampling for Subject {isub}, Region {visualRegion} ---")

    # Output file h5 file for saving processed pRF samples for a subject and ROI group, and log for processed image tracking
    h5_filename = os.path.join(
        prf_folder, f"prfSampleStim_v{visualRegion}_sub{isub}.h5"
    )
    # Define the path to the log file that keeps track of which images were already processed
    log_path = os.path.join(prf_folder, f"processed_images_sub{isub}.log")
    # Load previously processed image IDs from the log file (if it exists)
    processed_images = load_processed_images_log(log_path)

    # Load NSD experimental design data
    nsdDesignPath = os.path.join(
        nsd_folder, "experiments", "nsd_expdesign.mat"
    )  # Path to the experimental design MATLAB file that specifies image presentation order
    if not os.path.exists(
        nsdDesignPath
    ):  # Safety check: stop execution if the design file doesn't exist
        print(f"ERROR: NSD Design file not found: {nsdDesignPath}")
        return

    # Load the .mat file contents using scipy.io
    nsdDesign = sio.loadmat(nsdDesignPath)
    # subjectim: matrix of shape [n_subjects, n_trials] with image IDs for each subject and trial
    subjectim = nsdDesign["subjectim"]
    # masterordering: global order of trials (flattened and converted to 0-based indexing)
    masterordering = nsdDesign["masterordering"].flatten() - 1
    # Keep only those indices that are valid (i.e., do not exceed the number of trials for this subject)
    valid_masterordering = masterordering[masterordering < subjectim.shape[1]]
    allImgs = np.unique(
        subjectim[isub - 1, valid_masterordering]
    )  # allImgs: the unique set of image IDs shown to the subject across the valid trials

    # ------------------------------------------------------------
    # Limit processing to a small number of images when in test mode
    # ------------------------------------------------------------
    if test_mode:
        allImgs = allImgs[
            :5
        ]  # Only use the first 5 images to allow for faster debugging

    print(f"Total images to process: {len(allImgs)}")

    # ------------------------------------------------------------
    # Load subject-specific pRF maps and ROI labels
    # ------------------------------------------------------------

    # Load pRF maps and ROI definitions
    betas_folder = os.path.join(
        nsd_folder, f"subj{isub:02d}", "func1pt8mm"
    )  # Path to subject's beta folder (includes pRF maps and ROI data)
    prf_files = [
        "prf_angle.nii.gz",  # Preferred polar angle map
        "prf_eccentricity.nii.gz",  # Eccentricity (distance from fixation)
        "prf_size.nii.gz",  # Receptive field size
        "R2.nii.gz",  # Model fit quality (variance explained)
    ]
    # Load each pRF map into memory as a 3D numpy array
    angData, eccData, sizeData, r2Data = [
        load_nifti(os.path.join(betas_folder, f)) for f in prf_files
    ]
    # Load visual ROI labels: identifies each voxel's visual area (V1, V2, etc.)
    visRoiData = load_nifti(os.path.join(betas_folder, "roi", "prf-visualrois.nii.gz"))

    # If any of the pRF or ROI files failed to load, exit early
    if any(data is None for data in [angData, eccData, sizeData, r2Data, visRoiData]):
        print("ERROR: Missing pRF data. Check file paths.")
        return

    # ------------------------------------------------------------
    # Define which ROIs to include based on visualRegion group
    # ------------------------------------------------------------

    # Each visualRegion value corresponds to a group of ROIs:
    # e.g., visualRegion=1 includes V1v and V1d (indices 0 and 1)
    roi_map = {
        1: [0, 1],  # V1v + V1d
        2: [2, 3],  # V2v + V2d
        3: [4, 5],  # V3v + V3d
        4: [6],  # hV4
    }
    # Select the list of ROI indices for the current visualRegion
    rois = roi_map.get(visualRegion, [])

    # ------------------------------------------------------------
    # Setup output containers and load cached Gaussians if available
    # ------------------------------------------------------------

    # Path to the HDF5 file containing cached 2D Gaussian filters for this subject and region
    gauss_file = os.path.join(prf_folder, f"roiGaussians_sub{isub}_v{visualRegion}.h5")
    # Initialize containers:
    roiGaus = {}  # Maps ROI index → 3D array of Gaussian filters (voxels, H, W)
    roiPrf = {}  # Stores the pRF parameters for each voxel
    prfSampleLev = (
        {}
    )  # Output: per-voxel projection from orientation-summed pyramid levels
    prfSampleLevOri = {}  # Output: per-voxel projections per level and orientation
    missing_rois = []  # ROIs that don’t exist in the cache and must be computed

    # ------------------------------------------------------------
    # Load cached Gaussian pRF kernels if available
    # ------------------------------------------------------------
    if os.path.exists(gauss_file):
        print(f"Loading cached Gaussians from {gauss_file}")
        with h5py.File(gauss_file, "r") as f:
            for roinum in rois:
                key = f"roi_{roinum}"  # Dataset key inside HDF5 file
                if (
                    key in f
                ):  # Load precomputed 2D Gaussian kernels for all voxels in this ROI
                    G_all = f[key][:]
                    roiGaus[roinum] = G_all

                    # Extract pRF parameters for voxels in this ROI
                    roi_mask = visRoiData == (roinum + 1)
                    ecc = eccData[roi_mask]  # Eccentricity per voxel
                    ang = angData[roi_mask]  # Preferred angle per voxel
                    sz = sizeData[roi_mask] * pix_per_deg  # Convert size to pixels
                    x0 = ecc * np.cos(
                        np.deg2rad(ang) * 2
                    )  # x-position of Gaussian center
                    y0 = ecc * np.sin(
                        np.deg2rad(ang) * 2
                    )  # y-position of Gaussian center

                    # Store pRF metadata per voxel
                    roiPrf[roinum] = {
                        "ecc": ecc,
                        "ang": ang,
                        "sz": sz,
                        "x": x0,
                        "y": y0,
                    }

                    # Allocate output arrays to store projection results per voxel
                    prfSampleLev[roinum] = np.zeros(
                        (len(allImgs), len(ecc), num_levels + 2)
                    )
                    prfSampleLevOri[roinum] = np.zeros(
                        (len(allImgs), len(ecc), num_levels + 2, num_orientations)
                    )
                else:  # If the ROI data is missing from the cache, mark it for computation
                    missing_rois.append(roinum)
    else:  # If no cache file exists, all ROIs need to be computed
        missing_rois = rois

    # ------------------------------------------------------------
    # Compute missing Gaussian kernels for ROIs not found in cache
    # ------------------------------------------------------------

    if missing_rois:
        print(f"Computing Gaussians for missing ROIs: {missing_rois}")

        # Generate coordinate grid over stimulus image (512x512 grid)
        X, Y = np.meshgrid(
            np.linspace(
                -background_size * img_scaling / 2,
                background_size * img_scaling / 2,
                512,
            ),
            np.linspace(
                -background_size * img_scaling / 2,
                background_size * img_scaling / 2,
                512,
            ),
        )

        with h5py.File(gauss_file, "a") as f:  # Open HDF5 file in append mode
            for roinum in missing_rois:
                roi_mask = visRoiData == (roinum + 1)
                if np.sum(roi_mask) == 0:
                    print(f"Skipping empty ROI: {roinum}")
                    continue

                # Extract pRF parameters for voxels in this ROI
                ecc = eccData[roi_mask]
                ang = angData[roi_mask]
                sz = sizeData[roi_mask] * pix_per_deg
                x0 = ecc * np.cos(np.deg2rad(ang) * 2)
                y0 = ecc * np.sin(np.deg2rad(ang) * 2)

                roiPrf[roinum] = {"ecc": ecc, "ang": ang, "sz": sz, "x": x0, "y": y0}

                # Allocate result containers for this ROI
                prfSampleLev[roinum] = np.zeros(
                    (len(allImgs), len(ecc), num_levels + 2)
                )
                prfSampleLevOri[roinum] = np.zeros(
                    (len(allImgs), len(ecc), num_levels + 2, num_orientations)
                )

                # ------------------------------------------------
                # Create 2D Gaussian kernel for each voxel in ROI
                # ------------------------------------------------

                # Subtract center (x0, y0) from full 2D grid
                X_diff = X.reshape(1, 512, 512) - x0[:, None, None]
                Y_diff = Y.reshape(1, 512, 512) - y0[:, None, None]

                # Compute 2D isotropic Gaussian: G = exp(-((x−x0)² + (y−y0)²)/(2σ²))
                G_all = np.exp(-(X_diff**2 + Y_diff**2) / (2 * sz[:, None, None] ** 2))

                # Store Gaussian kernels in memory and cache them to HDF5 file
                roiGaus[roinum] = G_all
                f.create_dataset(f"roi_{roinum}", data=G_all, compression="gzip")

        print("Updated Gaussian cache with missing ROIs")

    # ------------------------------------------------------------
    # Image batch preparation
    # ------------------------------------------------------------

    work_list = []  # Final list of image numbers to be processed
    image_index_map = {}  # Mapping from image number to row index in result arrays
    kept_imgs = []  # List of image numbers that passed filtering and will be saved

    for iimg, imgNum in enumerate(allImgs):
        # Skip images already processed and listed in the log file
        if imgNum in processed_images:
            print(f"Skipping already processed image {imgNum}")
            continue

        # Check if the corresponding pyramid file exists
        pyramid_filename = os.path.join(pyramid_folder, f"pyrImg{iimg}.mat")
        if not os.path.exists(pyramid_filename):
            print(f" Missing pyramid file: {pyramid_filename}")
            continue

        # Add image to processing list and store its index
        work_list.append(imgNum)
        image_index_map[imgNum] = len(kept_imgs)
        kept_imgs.append(imgNum)

    # Exit early if all images were already processed or missing
    if not work_list:
        print("\n No new images to process. Skipping save.")
        return

    print(f"\nProcessing {len(work_list)} new images")

    # Divide images into batches
    num_batches = len(work_list) // batch_size + (
        1 if len(work_list) % batch_size != 0 else 0
    )

    # ------------------------------------------------------------
    # Process images in batches
    # ------------------------------------------------------------

    for batch_num in range(num_batches):
        # Extract image IDs for this batch
        batch = work_list[batch_num * batch_size : (batch_num + 1) * batch_size]
        print(
            f"Processing batch {batch_num + 1}/{num_batches} with {len(batch)} images"
        )

        if not batch:
            print(f"Skipping empty batch {batch_num + 1}/{num_batches}")
            continue

        # Run processing in parallel using joblib and show a progress bar with tqdm
        results = Parallel(n_jobs=5)(
            delayed(process_single_image)(imgNum, roiGaus, num_levels, num_orientations)
            for imgNum in tqdm(
                batch,
                desc=f"Batch {batch_num + 1}/{num_batches}",
                unit="image",
                ncols=100,
            )
        )

        # Store results in the output arrays for each ROI
        for imgNum, partial_lev, partial_ori in results:
            iimg = image_index_map[imgNum]
            for roinum in partial_lev:
                prfSampleLev[roinum][iimg, :, :] = partial_lev[roinum]
                prfSampleLevOri[roinum][iimg, :, :, :] = partial_ori[roinum]
            # Free memory after each image
            del partial_lev, partial_ori
            gc.collect()

        # ------------------------------------------------------------
        # Save results safely using a temporary HDF5 file
        # ------------------------------------------------------------

        with tempfile.NamedTemporaryFile(
            delete=False, suffix=".h5", dir=prf_folder
        ) as tmp_file:
            tmp_filename = tmp_file.name

        with h5py.File(tmp_filename, "w") as f:
            for roinum in prfSampleLev:
                # Save level-summed and orientation-specific data for each ROI
                f.create_dataset(
                    f"prfSampleLev/roi_{roinum}",
                    data=prfSampleLev[roinum],
                    compression="gzip",
                )
                f.create_dataset(
                    f"prfSampleLevOri/roi_{roinum}",
                    data=prfSampleLevOri[roinum],
                    compression="gzip",
                )

            # Save metadata and attributes
            f.create_dataset("allImgs", data=np.array(kept_imgs, dtype=np.int32))
            f.create_dataset("rois", data=np.array(rois, dtype=np.int32))
            f.attrs["numLevels"] = num_levels
            f.attrs["numOrientations"] = num_orientations
            f.attrs["interpImgSize"] = interp_img_size
            f.attrs["backgroundSize"] = background_size
            f.attrs["pixPerDeg"] = pix_per_deg

        # Atomically replace old file with the new file to avoid corruption
        shutil.move(tmp_filename, h5_filename)
        print(f"Batch {batch_num + 1} processed and saved")

        # ------------------------------------------------------------
        # Update log file with newly processed image numbers
        # ------------------------------------------------------------

        new_logged_imgs = [str(img) for img in batch if img not in processed_images]
        if new_logged_imgs:
            with open(log_path, "a") as log_file:
                log_file.write("\n".join(new_logged_imgs) + "\n")
            print(f"Logged {len(new_logged_imgs)} new images")


# ------------------------------------------------------------
# Script entry point — ensures proper execution context
# ------------------------------------------------------------
if __name__ == "__main__":
    from multiprocessing import freeze_support

    # On Windows, multiprocessing requires freeze_support() to safely spawn subprocesses.
    # This prevents errors when the script is run as a standalone module.
    freeze_support()  # Optional but recommended for Windows compatibility

    # Call the main function to run pRF sampling for subject 1 and visualRegion 1.
    # You can modify the arguments (subject index, region index, batch size) as needed.
    prf_sample_model_expand(isub=1, visualRegion=1, batch_size=5)

Check prfSampleStim_v1_sub1.h5 is valid

In [ ]:
import h5py

# Path to your file
file_path = r"C:\nsd\prfsample_expand\prfSampleStim_v1_sub1.h5"

with h5py.File(file_path, "r") as f:
    for key in f.keys():
        print("Top-level key:", key)
        try:
            print("  Subkeys:", list(f[key].keys()))
        except Exception as e:
            print("  Failed to read subkeys:", e)